In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

sequence = "I've been waiting for a HuggingFace course my whole life."

tokens = tokenizer.tokenize(sequence)

ids = tokenizer.convert_tokens_to_ids(tokens)

input_ids = torch.tensor([ids])

# tool ong sentence, needto be padding
print("IDs: ", input_ids)


output = model(input_ids)

print("\n")
print("Logits: ", output.logits)


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 24220.30it/s]

IDs:  tensor([[ 1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,  2607,
          2026,  2878,  2166,  1012]])


Logits:  tensor([[-2.7276,  2.8789]], grad_fn=<AddmmBackward0>)


In [5]:
# 1. Pourquoi le padding est necessaire
# la 2e phrase, plus courte
sequence2 = "I've been waiting."
tokens2 = tokenizer.tokenize(sequence2)
ids2 = tokenizer.convert_tokens_to_ids(tokens2)

print("longueur phrase 1 :", len(ids))
print("longueur phrase 2 :", len(ids2))

try:
    torch.tensor([ids, ids2])
except ValueError as e:
    print("ERREUR :", e)

longueur phrase 1 : 14
longueur phrase 2 : 6
ERREUR : expected sequence of length 14 at dim 1 (got 6)


In [17]:
# 2. Padding manuel
padding_id = tokenizer.pad_token_id
nb_padding = len(ids) - len(ids2)

print("pad_token :", tokenizer.pad_token, "| pad_token_id :", padding_id)
print("nb de tokens de padding a ajouter :", nb_padding)

ids2_pad = ids2 + [padding_id] * nb_padding

print("\nids: ", ids, " | longeur: ", len(ids))
print("ids2 AVANT padding :", ids2, "| longeur: ", len(ids2))
print("ids2 APRES padding :", ids2_pad, "| longeur: ", len(ids2_pad))

pad_token : [PAD] | pad_token_id : 0
nb de tokens de padding a ajouter : 8

ids:  [1045, 1005, 2310, 2042, 3403, 2005, 1037, 17662, 12172, 2607, 2026, 2878, 2166, 1012]  | longeur:  14
ids2 AVANT padding : [1045, 1005, 2310, 2042, 3403, 1012] | longeur:  6
ids2 APRES padding : [1045, 1005, 2310, 2042, 3403, 1012, 0, 0, 0, 0, 0, 0, 0, 0] | longeur:  14


In [18]:
# 3. Le batch est maintenant rectangulaire
batched_ids = torch.tensor([ids, ids2_pad])

print("shape :", batched_ids.shape)
print(batched_ids)

# version lisible : on voit les [PAD]
print("\ntokens ligne 2 :", tokenizer.convert_ids_to_tokens(batched_ids[1]))

shape : torch.Size([2, 14])
tensor([[ 1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,  2607,
          2026,  2878,  2166,  1012],
        [ 1045,  1005,  2310,  2042,  3403,  1012,     0,     0,     0,     0,
             0,     0,     0,     0]])

tokens ligne 2 : ['i', "'", 've', 'been', 'waiting', '.', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']


In [8]:
# 4. L'attention mask : des 0 pile sur les [PAD]
attention_mask = torch.tensor([
    [1] * len(ids),                       # phrase 1 : tout est reel
    [1] * len(ids2) + [0] * nb_padding,   # phrase 2 : reels puis padding
])

print(attention_mask)
print("\nnb de tokens reels par phrase :", attention_mask.sum(dim=1).tolist())

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]])

nb de tokens reels par phrase : [14, 6]


In [9]:
# 5. La preuve : sans le mask, les logits sont faux
print("phrase 2 SEULE           :", model(torch.tensor([ids2])).logits[0].tolist())
print("batch SANS mask, ligne 2 :", model(batched_ids).logits[1].tolist())
print("batch AVEC mask, ligne 2 :", model(batched_ids, attention_mask=attention_mask).logits[1].tolist())

phrase 2 SEULE           : [1.9874591827392578, -1.511502981185913]
batch SANS mask, ligne 2 : [1.043083906173706, -0.8302785754203796]
batch AVEC mask, ligne 2 : [1.9874567985534668, -1.5115007162094116]
